<a href="https://colab.research.google.com/github/saacosta-bit/Sinanzas-Agente/blob/main/notebooks/sinanzas_agente.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [40]:
import os
from google.colab import userdata

os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

In [41]:
!pip install -q langchain-google-genai langgraph faiss-cpu langchain-community langchain-text-splitters

In [42]:
import sqlite3
from datetime import datetime
from zoneinfo import ZoneInfo
from typing import Optional
from pydantic import BaseModel, Field
from langchain_core.tools import tool

DB_PATH = "sinanzas.db"
CATEGORIAS = ["Alimentación", "Transporte", "Servicios",
              "Entretenimiento", "Salud", "Educación", "Otros"]

def hoy_chile() -> str:
    return datetime.now(ZoneInfo("America/Santiago")).date().isoformat()

def init_db():
    with sqlite3.connect(DB_PATH) as conn:
        conn.execute("""
            CREATE TABLE IF NOT EXISTS gastos (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                monto REAL NOT NULL,
                categoria TEXT NOT NULL,
                descripcion TEXT,
                fecha TEXT NOT NULL
            )""")

init_db()

class GastoInput(BaseModel):
    monto: float = Field(gt=0, description="Monto del gasto en pesos chilenos")
    categoria: str = Field(description=f"Categoría, una de: {', '.join(CATEGORIAS)}")
    descripcion: str = Field(default="", description="Detalle breve del gasto")
    fecha: Optional[str] = Field(default=None,
        description="Fecha en formato YYYY-MM-DD. Si no se indica, se usa hoy")

@tool("registrar_gasto", args_schema=GastoInput)
def registrar_gasto(monto: float, categoria: str,
                    descripcion: str = "", fecha: Optional[str] = None) -> str:
    """Registra un gasto del usuario en la base de datos.
    Úsala cuando el usuario cuente que gastó dinero en algo."""
    if categoria not in CATEGORIAS:
        categoria = "Otros"
    try:
        fecha = datetime.strptime(fecha, "%Y-%m-%d").date().isoformat() if fecha else hoy_chile()
    except ValueError:
        return "Error: la fecha debe tener formato YYYY-MM-DD."
    with sqlite3.connect(DB_PATH) as conn:
        conn.execute(
            "INSERT INTO gastos (monto, categoria, descripcion, fecha) VALUES (?, ?, ?, ?)",
            (monto, categoria, descripcion, fecha))
    return f"Gasto registrado: ${monto:,.0f} en {categoria} ({fecha})."

In [43]:
print(registrar_gasto.invoke({"monto": 5000, "categoria": "Alimentación",
                              "descripcion": "Almuerzo"}))

with sqlite3.connect(DB_PATH) as conn:
    print(conn.execute("SELECT * FROM gastos").fetchall())

Gasto registrado: $5,000 en Alimentación (2026-10-20).
[(1, 5000.0, 'Alimentación', 'Almuerzo', '2026-10-20'), (2, 5000.0, 'Alimentación', 'Almuerzo', '2026-10-20'), (3, 5000.0, 'Alimentación', 'Almuerzo', '2026-10-20'), (4, 5000.0, 'Alimentación', 'Almuerzo', '2026-10-20'), (5, 5000.0, 'Alimentación', 'Almuerzo', '2026-10-20'), (6, 5000.0, 'Alimentación', 'Almuerzo', '2026-10-20')]


In [44]:
def init_tabla_ingresos():
    with sqlite3.connect(DB_PATH) as conn:
        conn.execute("""
            CREATE TABLE IF NOT EXISTS ingresos (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                monto REAL NOT NULL,
                fuente TEXT,
                fecha TEXT NOT NULL
            )""")

init_tabla_ingresos()

class IngresoInput(BaseModel):
    monto: float = Field(gt=0, description="Monto del ingreso en pesos chilenos")
    fuente: str = Field(default="Sueldo", description="Origen del ingreso (sueldo, honorarios, venta, etc.)")
    fecha: Optional[str] = Field(default=None,
        description="Fecha en formato YYYY-MM-DD. Si no se indica, se usa hoy")

@tool("registrar_ingreso", args_schema=IngresoInput)
def registrar_ingreso(monto: float, fuente: str = "Sueldo",
                      fecha: Optional[str] = None) -> str:
    """Registra un ingreso de dinero del usuario.
    Úsala cuando el usuario cuente cuánto ganó o recibió."""
    try:
        fecha = datetime.strptime(fecha, "%Y-%m-%d").date().isoformat() if fecha else hoy_chile()
    except ValueError:
        return "Error: la fecha debe tener formato YYYY-MM-DD."
    with sqlite3.connect(DB_PATH) as conn:
        conn.execute(
            "INSERT INTO ingresos (monto, fuente, fecha) VALUES (?, ?, ?)",
            (monto, fuente, fecha))
    return f"Ingreso registrado: ${monto:,.0f} por {fuente} ({fecha})."

In [45]:
def init_tabla_metas():
    with sqlite3.connect(DB_PATH) as conn:
        conn.execute("""
            CREATE TABLE IF NOT EXISTS metas (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                nombre TEXT NOT NULL,
                monto_objetivo REAL NOT NULL,
                plazo_meses INTEGER NOT NULL,
                fecha_inicio TEXT NOT NULL
            )""")

init_tabla_metas()

class MetaInput(BaseModel):
    nombre: str = Field(description="Nombre de la meta (ej. 'Notebook nuevo')")
    monto_objetivo: float = Field(gt=0, description="Monto total a ahorrar en pesos chilenos")
    plazo_meses: int = Field(gt=0, description="Plazo en meses para lograr la meta")

@tool("crear_meta_ahorro", args_schema=MetaInput)
def crear_meta_ahorro(nombre: str, monto_objetivo: float, plazo_meses: int) -> str:
    """Crea una meta de ahorro del usuario.
    Úsala cuando el usuario diga que quiere ahorrar una cantidad en un plazo."""
    with sqlite3.connect(DB_PATH) as conn:
        conn.execute(
            "INSERT INTO metas (nombre, monto_objetivo, plazo_meses, fecha_inicio) VALUES (?, ?, ?, ?)",
            (nombre, monto_objetivo, plazo_meses, hoy_chile()))
    mensual = monto_objetivo / plazo_meses
    return (f"Meta creada: '{nombre}' por ${monto_objetivo:,.0f} en {plazo_meses} meses "
            f"(equivale a ${mensual:,.0f} por mes).")

In [46]:
print(registrar_ingreso.invoke({"monto": 800000, "fuente": "Sueldo"}))
print(crear_meta_ahorro.invoke({"nombre": "Notebook nuevo",
                                "monto_objetivo": 600000, "plazo_meses": 6}))

with sqlite3.connect(DB_PATH) as conn:
    print(conn.execute("SELECT * FROM ingresos").fetchall())
    print(conn.execute("SELECT * FROM metas").fetchall())

Ingreso registrado: $800,000 por Sueldo (2026-10-20).
Meta creada: 'Notebook nuevo' por $600,000 en 6 meses (equivale a $100,000 por mes).
[(1, 800000.0, 'Sueldo', '2026-10-20'), (2, 800000.0, 'Sueldo', '2026-10-20'), (3, 800000.0, 'Sueldo', '2026-10-20'), (4, 800000.0, 'Sueldo', '2026-10-20'), (5, 800000.0, 'Sueldo', '2026-10-20'), (6, 800000.0, 'Sueldo', '2026-10-20')]
[(1, 'Notebook nuevo', 600000.0, 6, '2026-10-20'), (2, 'Notebook nuevo', 600000.0, 6, '2026-10-20'), (3, 'Notebook nuevo', 600000.0, 6, '2026-10-20'), (4, 'Notebook nuevo', 600000.0, 6, '2026-10-20'), (5, 'Notebook nuevo', 600000.0, 6, '2026-10-20'), (6, 'Notebook nuevo', 600000.0, 6, '2026-10-20')]


In [47]:
class ConsultaInput(BaseModel):
    categoria: Optional[str] = Field(default=None,
        description=f"Categoría a filtrar (opcional): {', '.join(CATEGORIAS)}")
    mes: Optional[str] = Field(default=None,
        description="Mes a consultar en formato YYYY-MM (opcional). Si no se indica, trae todos")

@tool("consultar_gastos", args_schema=ConsultaInput)
def consultar_gastos(categoria: Optional[str] = None, mes: Optional[str] = None) -> str:
    """Consulta los gastos registrados, con filtro opcional por categoría y mes.
    Devuelve el detalle y el total. Úsala cuando el usuario pregunte cuánto gastó."""
    query = "SELECT fecha, categoria, descripcion, monto FROM gastos WHERE 1=1"
    params = []
    if categoria:
        query += " AND categoria = ?"
        params.append(categoria)
    if mes:
        query += " AND substr(fecha, 1, 7) = ?"
        params.append(mes)
    query += " ORDER BY fecha DESC"

    with sqlite3.connect(DB_PATH) as conn:
        filas = conn.execute(query, params).fetchall()

    if not filas:
        return "No hay gastos registrados con esos filtros."
    total = sum(f[3] for f in filas)
    detalle = "\n".join(f"- {f[0]} | {f[1]} | {f[2] or 'sin descripción'} | ${f[3]:,.0f}" for f in filas)
    return f"{detalle}\nTotal: ${total:,.0f} ({len(filas)} gastos)"

In [48]:
!pip install -q faiss-cpu langchain-community langchain-text-splitters

from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

# CONTENIDO DE EJEMPLO: reemplázalo por guías reales y cita sus fuentes en el informe
GUIAS = [
    "Regla 50/30/20: destina 50% del ingreso a necesidades básicas, 30% a gustos personales y 20% a ahorro o pago de deudas.",
    "Gastos hormiga: son gastos pequeños y frecuentes, como cafés o snacks diarios, que sumados al mes pueden representar un monto importante. Anótalos para hacerlos visibles.",
    "Fondo de emergencia: apunta a reunir entre 3 y 6 meses de gastos básicos en una cuenta separada y de fácil acceso.",
    "Para salir de deudas, prioriza pagar primero la de mayor tasa de interés y mantén al día los pagos mínimos de las demás.",
    "Ahorra primero: transfiere un porcentaje fijo de tu ingreso a una cuenta de ahorro apenas lo recibes, en vez de ahorrar lo que sobre a fin de mes.",
    "Antes de una compra no planificada, espera 24 horas. Si sigues queriéndola, revisa si cabe en tu presupuesto del mes.",
]

embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks = splitter.create_documents(GUIAS)
vector_store = FAISS.from_documents(chunks, embeddings)

class GuiaInput(BaseModel):
    pregunta: str = Field(description="Consulta del usuario sobre finanzas personales")

@tool("buscar_guias_financieras", args_schema=GuiaInput)
def buscar_guias_financieras(pregunta: str) -> str:
    """Busca consejos en las guías de educación financiera.
    Úsala cuando el usuario pregunte cómo ahorrar, presupuestar o manejar deudas."""
    resultados = vector_store.similarity_search(pregunta, k=2)
    return "\n".join(f"- {doc.page_content}" for doc in resultados)

In [49]:
registrar_gasto.invoke({"monto": 2500, "categoria": "Alimentación", "descripcion": "Café"})
registrar_gasto.invoke({"monto": 1200, "categoria": "Transporte", "descripcion": "Metro"})

print(consultar_gastos.invoke({}))
print("---")
print(consultar_gastos.invoke({"categoria": "Alimentación", "mes": "2026-10"}))
print("---")
print(buscar_guias_financieras.invoke({"pregunta": "¿Cómo puedo ahorrar más dinero?"}))

- 2026-10-20 | Alimentación | Almuerzo | $5,000
- 2026-10-20 | Alimentación | Almuerzo | $5,000
- 2026-10-20 | Alimentación | Almuerzo | $5,000
- 2026-10-20 | Alimentación | Almuerzo | $5,000
- 2026-10-20 | Alimentación | Almuerzo | $5,000
- 2026-10-20 | Alimentación | Almuerzo | $5,000
- 2026-10-20 | Alimentación | Café | $2,500
- 2026-10-20 | Transporte | Metro | $1,200
Total: $33,700 (8 gastos)
---
- 2026-10-20 | Alimentación | Almuerzo | $5,000
- 2026-10-20 | Alimentación | Almuerzo | $5,000
- 2026-10-20 | Alimentación | Almuerzo | $5,000
- 2026-10-20 | Alimentación | Almuerzo | $5,000
- 2026-10-20 | Alimentación | Almuerzo | $5,000
- 2026-10-20 | Alimentación | Almuerzo | $5,000
- 2026-10-20 | Alimentación | Café | $2,500
Total: $32,500 (7 gastos)
---
- Ahorra primero: transfiere un porcentaje fijo de tu ingreso a una cuenta de ahorro apenas lo recibes, en vez de ahorrar lo que sobre a fin de mes.
- Regla 50/30/20: destina 50% del ingreso a necesidades básicas, 30% a gustos person

In [50]:
class HormigaInput(BaseModel):
    mes: Optional[str] = Field(default=None,
        description="Mes en formato YYYY-MM. Si se omite, se usa el mes actual")
    umbral_monto: float = Field(default=5000, gt=0,
        description="Monto máximo (CLP) para considerar un gasto como pequeño")
    min_repeticiones: int = Field(default=3, ge=2,
        description="Veces mínimas que debe repetirse el gasto en el mes")

@tool("detectar_gastos_hormiga", args_schema=HormigaInput)
def detectar_gastos_hormiga(mes: Optional[str] = None, umbral_monto: float = 5000,
                            min_repeticiones: int = 3) -> str:
    """Detecta gastos hormiga: gastos pequeños que se repiten muchas veces en el mes.
    Úsala cuando el usuario quiera saber en qué se le va el dinero en cosas pequeñas."""
    mes = mes or hoy_chile()[:7]
    with sqlite3.connect(DB_PATH) as conn:
        total_mes = conn.execute(
            "SELECT COALESCE(SUM(monto), 0) FROM gastos WHERE substr(fecha, 1, 7) = ?",
            (mes,)).fetchone()[0]
        filas = conn.execute("""
            SELECT categoria, LOWER(TRIM(descripcion)), COUNT(*), SUM(monto)
            FROM gastos
            WHERE substr(fecha, 1, 7) = ? AND monto <= ?
            GROUP BY categoria, LOWER(TRIM(descripcion))
            HAVING COUNT(*) >= ?
            ORDER BY SUM(monto) DESC""", (mes, umbral_monto, min_repeticiones)).fetchall()

    if not filas:
        return f"No se detectaron gastos hormiga en {mes}."
    suma = sum(f[3] for f in filas)
    pct = (suma / total_mes * 100) if total_mes else 0
    detalle = "\n".join(
        f"- {f[1] or 'sin descripción'} ({f[0]}): {f[2]} veces, ${f[3]:,.0f}" for f in filas)
    return (f"Gastos hormiga en {mes}:\n{detalle}\n"
            f"Total: ${suma:,.0f}, equivale al {pct:.1f}% de tus gastos del mes.")

In [51]:
import math

CATEGORIAS_FLEXIBLES = ["Entretenimiento", "Alimentación", "Otros"]

class PlanInput(BaseModel):
    nombre_meta: Optional[str] = Field(default=None,
        description="Nombre de la meta. Si se omite, se usa la más reciente")
    mes: Optional[str] = Field(default=None,
        description="Mes a analizar en formato YYYY-MM. Si se omite, el mes actual")

@tool("calcular_plan_ahorro", args_schema=PlanInput)
def calcular_plan_ahorro(nombre_meta: Optional[str] = None, mes: Optional[str] = None) -> str:
    """Calcula si el usuario puede cumplir una meta de ahorro con sus ingresos y gastos
    del mes, y sugiere recortes por categoría si no alcanza."""
    mes = mes or hoy_chile()[:7]
    with sqlite3.connect(DB_PATH) as conn:
        if nombre_meta:
            meta = conn.execute(
                "SELECT nombre, monto_objetivo, plazo_meses FROM metas "
                "WHERE LOWER(nombre) = LOWER(?) ORDER BY id DESC LIMIT 1", (nombre_meta,)).fetchone()
        else:
            meta = conn.execute(
                "SELECT nombre, monto_objetivo, plazo_meses FROM metas ORDER BY id DESC LIMIT 1").fetchone()
        ingresos = conn.execute(
            "SELECT COALESCE(SUM(monto), 0) FROM ingresos WHERE substr(fecha, 1, 7) = ?",
            (mes,)).fetchone()[0]
        por_cat = conn.execute(
            "SELECT categoria, SUM(monto) FROM gastos WHERE substr(fecha, 1, 7) = ? "
            "GROUP BY categoria ORDER BY SUM(monto) DESC", (mes,)).fetchall()

    if not meta:
        return "No hay metas de ahorro registradas. Crea una primero."
    if ingresos == 0:
        return f"No hay ingresos registrados en {mes}. Registra un ingreso para calcular el plan."

    nombre, objetivo, plazo = meta
    gastos = sum(m for _, m in por_cat)
    disponible = ingresos - gastos
    requerido = objetivo / plazo

    resumen = (f"Meta '{nombre}': ${objetivo:,.0f} en {plazo} meses (${requerido:,.0f} por mes). "
               f"Ingresos de {mes}: ${ingresos:,.0f}, gastos: ${gastos:,.0f}, "
               f"disponible: ${disponible:,.0f}.")
    if disponible >= requerido:
        return resumen + " La meta es alcanzable con tu ritmo actual."

    deficit = requerido - disponible
    flexibles = [(c, m) for c, m in por_cat if c in CATEGORIAS_FLEXIBLES]
    total_flex = sum(m for _, m in flexibles)
    sugerencias = []
    for c, m in flexibles:
        recorte = min(deficit * m / total_flex, m) if total_flex else 0
        sugerencias.append(f"- Reducir {c} en ${recorte:,.0f} (gastas ${m:,.0f})")
    texto = f"{resumen}\nTe faltan ${deficit:,.0f} al mes."
    if sugerencias:
        texto += "\nRecortes sugeridos:\n" + "\n".join(sugerencias)
    if total_flex < deficit:
        texto += "\nAun recortando todo lo flexible no alcanza."
    if disponible > 0:
        texto += f"\nAlternativa: ampliar el plazo a {math.ceil(objetivo / disponible)} meses."
    return texto

In [52]:
for _ in range(5):
    registrar_gasto.invoke({"monto": 2500, "categoria": "Alimentación", "descripcion": "Café"})
registrar_gasto.invoke({"monto": 300000, "categoria": "Servicios", "descripcion": "Arriendo"})
registrar_gasto.invoke({"monto": 250000, "categoria": "Alimentación", "descripcion": "Supermercado"})
registrar_gasto.invoke({"monto": 180000, "categoria": "Entretenimiento", "descripcion": "Salidas"})

print(detectar_gastos_hormiga.invoke({}))
print("---")
print(calcular_plan_ahorro.invoke({}))

Gastos hormiga en 2026-10:
- almuerzo (Alimentación): 6 veces, $30,000
- café (Alimentación): 6 veces, $15,000
Total: $45,000, equivale al 5.8% de tus gastos del mes.
---
Meta 'Notebook nuevo': $600,000 en 6 meses ($100,000 por mes). Ingresos de 2026-10: $4,800,000, gastos: $776,200, disponible: $4,023,800. La meta es alcanzable con tu ritmo actual.


In [53]:
def mes_anterior(mes: str) -> str:
    anio, m = map(int, mes.split("-"))
    return f"{anio - 1}-12" if m == 1 else f"{anio}-{m - 1:02d}"

def gasto_por_categoria(mes: str) -> dict:
    with sqlite3.connect(DB_PATH) as conn:
        return dict(conn.execute(
            "SELECT categoria, SUM(monto) FROM gastos "
            "WHERE substr(fecha, 1, 7) = ? GROUP BY categoria", (mes,)).fetchall())

class AlertaInput(BaseModel):
    categoria: Optional[str] = Field(default=None,
        description="Categoría a revisar. Si se omite, se revisan todas")
    mes: Optional[str] = Field(default=None,
        description="Mes en formato YYYY-MM. Si se omite, el mes actual")
    umbral_porcentaje: float = Field(default=30, gt=0,
        description="Aumento porcentual respecto al mes anterior que dispara la alerta")
    limite_monto: Optional[float] = Field(default=None, gt=0,
        description="Límite de gasto en pesos definido por el usuario (opcional)")

@tool("generar_alerta_categoria", args_schema=AlertaInput)
def generar_alerta_categoria(categoria: Optional[str] = None, mes: Optional[str] = None,
                             umbral_porcentaje: float = 30,
                             limite_monto: Optional[float] = None) -> str:
    """Genera alertas cuando el gasto de una categoría supera el límite del usuario
    o aumenta mucho respecto al mes anterior."""
    mes = mes or hoy_chile()[:7]
    prev = mes_anterior(mes)
    actual = gasto_por_categoria(mes)
    anterior = gasto_por_categoria(prev)
    categorias = [categoria] if categoria else list(actual.keys())

    alertas = []
    for c in categorias:
        g, p = actual.get(c, 0), anterior.get(c, 0)
        if p > 0 and g > p * (1 + umbral_porcentaje / 100):
            alertas.append(f"ALERTA en {c}: ${g:,.0f} en {mes}, "
                           f"{(g - p) / p * 100:.0f}% más que en {prev} (${p:,.0f}).")
        if limite_monto and g > limite_monto:
            alertas.append(f"ALERTA en {c}: superaste tu límite de ${limite_monto:,.0f} "
                           f"(llevas ${g:,.0f}).")
    if not alertas:
        return f"Sin alertas en {mes}: tus gastos están dentro de lo esperado."
    return "\n".join(alertas)

In [54]:
class ReporteInput(BaseModel):
    mes: Optional[str] = Field(default=None,
        description="Mes en formato YYYY-MM. Si se omite, el mes actual")

@tool("generar_reporte_mensual", args_schema=ReporteInput)
def generar_reporte_mensual(mes: Optional[str] = None) -> str:
    """Genera el resumen del mes: ingresos, gastos por categoría, ahorro
    y comparación con el mes anterior."""
    mes = mes or hoy_chile()[:7]
    prev = mes_anterior(mes)
    with sqlite3.connect(DB_PATH) as conn:
        ingresos = conn.execute(
            "SELECT COALESCE(SUM(monto), 0) FROM ingresos WHERE substr(fecha, 1, 7) = ?",
            (mes,)).fetchone()[0]
    por_cat = gasto_por_categoria(mes)
    gastos = sum(por_cat.values())
    gastos_prev = sum(gasto_por_categoria(prev).values())
    ahorro = ingresos - gastos

    lineas = [f"REPORTE {mes}",
              f"Ingresos: ${ingresos:,.0f}",
              f"Gastos: ${gastos:,.0f}",
              f"Ahorro: ${ahorro:,.0f}" +
              (f" ({ahorro / ingresos * 100:.1f}% del ingreso)" if ingresos else ""),
              "Gastos por categoría:"]
    for c, m in sorted(por_cat.items(), key=lambda x: -x[1]):
        lineas.append(f"- {c}: ${m:,.0f} ({m / gastos * 100:.1f}%)")
    if gastos_prev > 0:
        dif = (gastos - gastos_prev) / gastos_prev * 100
        lineas.append(f"Comparación con {prev}: gastos {dif:+.1f}% (${gastos_prev:,.0f} el mes anterior).")
    else:
        lineas.append(f"Sin datos de {prev} para comparar.")
    return "\n".join(lineas)

In [55]:
# Datos simulados del mes anterior
registrar_ingreso.invoke({"monto": 800000, "fuente": "Sueldo", "fecha": "2026-09-05"})
registrar_gasto.invoke({"monto": 300000, "categoria": "Servicios", "descripcion": "Arriendo", "fecha": "2026-09-05"})
registrar_gasto.invoke({"monto": 200000, "categoria": "Alimentación", "descripcion": "Supermercado", "fecha": "2026-09-10"})
registrar_gasto.invoke({"monto": 80000, "categoria": "Entretenimiento", "descripcion": "Salidas", "fecha": "2026-09-20"})

print(generar_alerta_categoria.invoke({}))
print("---")
print(generar_reporte_mensual.invoke({}))

ALERTA en Alimentación: $295,000 en 2026-10, 48% más que en 2026-09 ($200,000).
ALERTA en Entretenimiento: $180,000 en 2026-10, 125% más que en 2026-09 ($80,000).
---
REPORTE 2026-10
Ingresos: $4,800,000
Gastos: $776,200
Ahorro: $4,023,800 (83.8% del ingreso)
Gastos por categoría:
- Servicios: $300,000 (38.6%)
- Alimentación: $295,000 (38.0%)
- Entretenimiento: $180,000 (23.2%)
- Transporte: $1,200 (0.2%)
Comparación con 2026-09: gastos +33.8% ($580,000 el mes anterior).


In [57]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.messages import SystemMessage, HumanMessage

herramientas = [registrar_gasto, registrar_ingreso, crear_meta_ahorro,
                consultar_gastos, buscar_guias_financieras,
                detectar_gastos_hormiga, calcular_plan_ahorro,
                generar_alerta_categoria, generar_reporte_mensual]

llm_agente = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite").bind_tools(herramientas)

def extraer_texto_limpio(contenido) -> str:
    if isinstance(contenido, str):
        return contenido
    return "".join(p.get("text", "") if isinstance(p, dict) else str(p) for p in contenido)

def prompt_sistema() -> SystemMessage:
    return SystemMessage(content=(
        "Eres Sinanzas, un asistente de finanzas personales para usuarios en Chile. "
        "Responde siempre en español, de forma clara y breve. "
        "Usa las herramientas para registrar y consultar datos; nunca inventes cifras. "
        "Si falta un dato necesario (monto, categoría), pregúntalo antes de actuar. "
        f"La fecha de hoy es {hoy_chile()} (formato YYYY-MM-DD)."))

def nodo_agente(state: MessagesState):
    respuesta = llm_agente.invoke([prompt_sistema()] + state["messages"])
    return {"messages": [respuesta]}

grafo = StateGraph(MessagesState)
grafo.add_node("agente", nodo_agente)
grafo.add_node("herramientas", ToolNode(herramientas))
grafo.add_edge(START, "agente")
grafo.add_conditional_edges("agente", tools_condition,
                            {"tools": "herramientas", END: END})
grafo.add_edge("herramientas", "agente")

agente = grafo.compile(checkpointer=MemorySaver())

def chatear(texto: str, thread_id: str = "usuario_1") -> str:
    config = {"configurable": {"thread_id": thread_id}}
    res = agente.invoke({"messages": [HumanMessage(content=texto)]}, config=config)
    msgs = res["messages"]
    ult = max(i for i, m in enumerate(msgs) if isinstance(m, HumanMessage))
    for m in msgs[ult:]:
        for tc in getattr(m, "tool_calls", None) or []:
            print(f"[herramienta] {tc['name']}({tc['args']})")
    return extraer_texto_limpio(msgs[-1].content)

In [58]:
print(chatear("Hoy gasté $4.500 en un café"))
print("---")
print(chatear("Anota otro gasto igual, pero de ayer"))
print("---")
print(chatear("¿Cómo voy este mes? Dame el reporte"))

[herramienta] registrar_gasto({'descripcion': 'Café', 'monto': 4500, 'categoria': 'Alimentación'})
¡Listo! Registré tu gasto de $4.500 en **Alimentación** (Café) con fecha de hoy.
---
[herramienta] registrar_gasto({'monto': 4500, 'fecha': '2026-10-19', 'descripcion': 'Café', 'categoria': 'Alimentación'})
¡Listo! Registré tu gasto de $4.500 en **Alimentación** (Café) con fecha de ayer (19 de octubre de 2026).
---
[herramienta] generar_reporte_mensual({'mes': '2026-10'})
Aquí tienes el resumen de tus finanzas en **octubre de 2026**:

* **Ingresos:** $4.800.000
* **Gastos totales:** $785.200 *(incluyendo tus cafés)*
* **Ahorro:** $4.014.800 (83,6% de tus ingresos)

**Gastos por categoría:**
* Alimentación: $304.000 (38,7%)
* Servicios: $300.000 (38,2%)
* Entretenimiento: $180.000 (22,9%)
* Transporte: $1.200 (0,2%)

Tus gastos han aumentado un **35,4%** en comparación con septiembre ($580.000). ¿Te gustaría que revisemos si tienes algún gasto hormiga o si necesitas ajustar el presupuesto 

In [59]:
def cargar_datos_demo():
    """Borra los datos y carga un conjunto simulado coherente (septiembre y octubre)."""
    with sqlite3.connect(DB_PATH) as conn:
        for tabla in ("gastos", "ingresos", "metas"):
            conn.execute(f"DELETE FROM {tabla}")
        conn.execute("DELETE FROM sqlite_sequence")

    ingresos = [(800000, "Sueldo", "2026-09-05"), (800000, "Sueldo", "2026-10-05")]
    gastos = [
        # Septiembre
        (300000, "Servicios", "Arriendo", "2026-09-05"),
        (200000, "Alimentación", "Supermercado", "2026-09-10"),
        (80000, "Entretenimiento", "Salidas", "2026-09-20"),
        # Octubre
        (300000, "Servicios", "Arriendo", "2026-10-05"),
        (250000, "Alimentación", "Supermercado", "2026-10-10"),
        (180000, "Entretenimiento", "Salidas", "2026-10-15"),
        (1200, "Transporte", "Metro", "2026-10-12"),
    ] + [(2500, "Alimentación", "Café", f"2026-10-{d:02d}") for d in (6, 8, 12, 14, 16)]

    for monto, fuente, fecha in ingresos:
        registrar_ingreso.invoke({"monto": monto, "fuente": fuente, "fecha": fecha})
    for monto, cat, desc, fecha in gastos:
        registrar_gasto.invoke({"monto": monto, "categoria": cat, "descripcion": desc, "fecha": fecha})
    crear_meta_ahorro.invoke({"nombre": "Notebook nuevo", "monto_objetivo": 600000, "plazo_meses": 6})
    print("Datos de demo cargados.")

cargar_datos_demo()

Datos de demo cargados.


In [60]:
# --- Memoria de largo plazo: SQLite (fuente de verdad) + FAISS (búsqueda semántica) ---
class MemoriaLargoPlazo:
    def __init__(self):
        with sqlite3.connect(DB_PATH) as conn:
            conn.execute("""CREATE TABLE IF NOT EXISTS memorias (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                texto TEXT NOT NULL UNIQUE,
                fecha TEXT NOT NULL)""")
        self.embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")
        self.indice = None
        textos = self.listar()
        if textos:
            self.indice = FAISS.from_texts(textos, self.embeddings)

    def listar(self):
        with sqlite3.connect(DB_PATH) as conn:
            return [f[0] for f in conn.execute("SELECT texto FROM memorias ORDER BY id")]

    def guardar(self, texto: str) -> bool:
        with sqlite3.connect(DB_PATH) as conn:
            try:
                conn.execute("INSERT INTO memorias (texto, fecha) VALUES (?, ?)", (texto, hoy_chile()))
            except sqlite3.IntegrityError:
                return False  # ya existía
        if self.indice is None:
            self.indice = FAISS.from_texts([texto], self.embeddings)
        else:
            self.indice.add_texts([texto])
        return True

    def buscar(self, consulta: str, k: int = 3):
        if self.indice is None:
            return []
        return [d.page_content for d in self.indice.similarity_search(consulta, k=k)]

memoria_lp = MemoriaLargoPlazo()

class MemoriaInput(BaseModel):
    texto: str = Field(description="Dato duradero sobre el usuario, en una frase breve "
                                   "(preferencia, objetivo o situación financiera estable)")

@tool("guardar_memoria", args_schema=MemoriaInput)
def guardar_memoria(texto: str) -> str:
    """Guarda en la memoria de largo plazo un dato duradero del usuario.
    Úsala cuando cuente una preferencia, un objetivo o algo que valga recordar
    en futuras conversaciones. No la uses para registrar gastos."""
    return "Recuerdo guardado." if memoria_lp.guardar(texto) else "Ya lo tenía anotado."

# --- Agente con recuperación automática de contexto ---
class EstadoSinanzas(MessagesState):
    contexto: str

herramientas_v2 = herramientas + [guardar_memoria]
llm_agente_v2 = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite").bind_tools(herramientas_v2)

def nodo_recuperar_contexto(state: EstadoSinanzas):
    consulta = extraer_texto_limpio(state["messages"][-1].content)
    recuerdos = memoria_lp.buscar(consulta, k=3)
    return {"contexto": "\n".join(f"- {r}" for r in recuerdos)}

def nodo_agente_v2(state: EstadoSinanzas):
    contexto = state.get("contexto") or "(sin recuerdos relevantes)"
    sistema = SystemMessage(content=(
        prompt_sistema().content +
        f"\n\nRecuerdos del usuario (memoria de largo plazo):\n{contexto}\n"
        "Si el usuario cuenta una preferencia u objetivo duradero, guárdalo con guardar_memoria. "
        "No guardes datos sensibles (salud, claves, números de tarjeta) ni montos de gastos."))
    return {"messages": [llm_agente_v2.invoke([sistema] + state["messages"])]}

grafo2 = StateGraph(EstadoSinanzas)
grafo2.add_node("recuperar_contexto", nodo_recuperar_contexto)
grafo2.add_node("agente", nodo_agente_v2)
grafo2.add_node("herramientas", ToolNode(herramientas_v2))
grafo2.add_edge(START, "recuperar_contexto")
grafo2.add_edge("recuperar_contexto", "agente")
grafo2.add_conditional_edges("agente", tools_condition, {"tools": "herramientas", END: END})
grafo2.add_edge("herramientas", "agente")

agente = grafo2.compile(checkpointer=MemorySaver())  # reemplaza al agente anterior

In [61]:
# Conversación A: el usuario cuenta datos duraderos
print(chatear("Me pagan el día 5 de cada mes y quiero ahorrar para un viaje a Japón. "
              "Prefiero ahorros pequeños y graduales.", thread_id="conversacion_A"))
print("---")

# Conversación B: hilo nuevo, sin memoria de corto plazo
print(chatear("¿Cuándo me pagan y para qué estoy ahorrando?", thread_id="conversacion_B"))
print("---")
print("Recuerdos guardados:", memoria_lp.listar())

[herramienta] guardar_memoria({'texto': 'Al usuario le pagan el 5 de cada mes y su objetivo es ahorrar para un viaje a Japón prefiriendo ahorros pequeños y graduales.'})
¡Excelente objetivo! Viajar a Japón es un gran proyecto. 

Para ayudarte a armar un plan a tu medida con ahorros pequeños y graduales, necesito saber dos cosas:
1. ¿Cuánto te gustaría juntar en total (o cuánto estimas que costará el viaje)?
2. ¿En cuántos meses te gustaría cumplir esta meta?
---
Te pagan el **5 de cada mes** y estás ahorrando para un **viaje a Japón**, prefiriendo ahorros pequeños y graduales.
---
Recuerdos guardados: ['Al usuario le pagan el 5 de cada mes y su objetivo es ahorrar para un viaje a Japón prefiriendo ahorros pequeños y graduales.']


In [65]:
print("--- PRUEBA 1: Crear meta ---")
print(chatear("Quiero crear una meta de ahorro para un viaje a Japón", thread_id="prueba_plan"))

print("\n--- PRUEBA 2: Dar los datos de la meta ---")
print(chatear("Quiero ahorrar 1.000.000 en 6 meses", thread_id="prueba_plan"))

print("\n--- PRUEBA 3: Evaluar la meta (Planificación) ---")
print(chatear("¿Puedo cumplir esa meta con mis ingresos actuales?", thread_id="prueba_plan"))

--- PRUEBA 1: Crear meta ---
[herramienta] guardar_memoria({'texto': 'El usuario tiene como objetivo de ahorro un viaje a Japón prefiriendo ahorros pequeños y graduales.'})
¡Excelente meta! Un viaje a Japón es un proyecto maravilloso. 

Para poder crear la meta de ahorro, ¿me podrías indicar **cuánto dinero necesitas juntar** (en pesos chilenos) y **en cuántos meses** te gustaría lograrlo?

--- PRUEBA 2: Dar los datos de la meta ---
[herramienta] crear_meta_ahorro({'monto_objetivo': 1000000, 'plazo_meses': 6, 'nombre': 'Viaje a Japón'})
¡Listo! He creado tu meta de ahorro:

* **Meta:** Viaje a Japón
* **Monto:** $1.000.000
* **Plazo:** 6 meses ($166.667 mensuales)

Como prefieres un enfoque gradual, ¿te gustaría que calculemos si este monto mensual se ajusta a tus ingresos y gastos actuales, o prefieres revisar algún recorte en tus gastos hormiga?

--- PRUEBA 3: Evaluar la meta (Planificación) ---
[herramienta] calcular_plan_ahorro({'nombre_meta': 'Viaje a Japón'})
Actualmente tienes i